<!-- mccl/notebooks/curvature_generalization.ipynb -->
# Courbure et généralisation

**Objectif :** mesurer l'impact de la courbure négative (variété hyperbolique) sur la généralisation.

**Protocole :**
1. Chargement des données (dataset2.zip)
2. Construction de deux modèles identiques sauf sur la tête variété
   - **Euclidien** : tête R^192, aucune courbure
   - **Courbe** : tête E×H×S, courbure négative sur H
3. Entraînement sur les 5 classes (mêmes hyperparamètres)
4. Évaluation in-distribution (test)
5. Évaluation OOD (corruptions ou second jeu de données)
6. Comparaison et figures

**Hypothèse testée :** la courbure négative améliore la généralisation sur les décalages covariables (corruptions, conditions différentes).

## 1. Environnement et imports

In [3]:
import sys
from pathlib import Path
import copy
import pickle
import zipfile

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

from src.config import ModelCfg, ManifoldCfg
from src.models.jepa import JepaModel
from src.training.optim import build_optimizer
from src.training.losses import classification_loss
from src.training.trainer import train_one_task
from src.evaluation.metrics import per_class_accuracy
from src.utils.seed import set_seed
from src.utils.logging import get_logger

set_seed(0)
log = get_logger("curv")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT   :", ROOT)
print("device :", device)
print("torch  :", torch.__version__)

ROOT   : c:\Users\hp\Documents\mccl
device : cpu
torch  : 2.14.1+cpu


## 2. Chargement des données

Lecture du `loader.pkl` contenu dans `dataset2.zip`. Restauration de `data_` depuis `original_data_` (mis à `None` par `Load_data.save()`).

In [4]:
ZIP_PATH = ROOT / "dataset2.zip"
if not ZIP_PATH.exists():
    ZIP_PATH = ROOT / "data" / "dataset2.zip"
INTERNAL = "loader.pkl"

print("Chargement :", ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    with zf.open(INTERNAL) as f:
        loader = pickle.load(f, encoding="latin1")

if loader.data_ is None:
    loader.data_ = copy.deepcopy(loader.original_data_)
    print("data_ restauré depuis original_data_")

real_df = loader.data_
print("Shape   :", real_df.shape)
print("Classes :", loader.name_label_)

if not hasattr(loader, "class_mapping_") or not loader.class_mapping_:
    loader.class_mapping_ = {c: i for i, c in enumerate(loader.name_label_)}

if not hasattr(loader, "image_shape_") or loader.image_shape_ is None:
    h, w, c = np.asarray(real_df["Image"].iloc[0]).shape
    loader.image_shape_ = (int(h), int(w), int(c))

print("Mapping :", loader.class_mapping_)
print("Shape   :", loader.image_shape_)

Chargement : c:\Users\hp\Documents\mccl\dataset2.zip
data_ restauré depuis original_data_
Shape   : (12138, 2)
Classes : ['Canette', 'Organique', 'Plastique', 'Textile', 'Verre']
Mapping : {'Canette': 0, 'Organique': 1, 'Plastique': 2, 'Textile': 3, 'Verre': 4}
Shape   : (240, 240, 3)


## 3. Splits train / val / test

Split 70/15/15 stratifié par classe.

In [5]:
from sklearn.model_selection import train_test_split
from src.data.splits import save_splits

labels = real_df["Label"].map(loader.class_mapping_).to_numpy(dtype=int)
n_classes = len(loader.name_label_)
idx = np.arange(len(real_df))

tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=0)
va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=0)

print(f"train={len(tr)} | val={len(va)} | test={len(te)}")
print("Counts train :", np.bincount(labels[tr]))
print("Counts val   :", np.bincount(labels[va]))
print("Counts test  :", np.bincount(labels[te]))

save_splits(str(ROOT / "data" / "splits" / "waste5.json"),
            {"train": tr, "val": va, "test": te})

train=8496 | val=1821 | test=1821
Counts train : [1254 1639 1808 1866 1929]
Counts val   : [269 351 387 400 414]
Counts test  : [269 352 388 399 413]


## 4. Transformations et Dataset

In [6]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

transform_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transform_eval = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ImageSubset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        img = self.images[j]
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.labels[j])


images = real_df["Image"].tolist()

ds_train = ImageSubset(images, labels, tr, transform_train)
ds_val   = ImageSubset(images, labels, va, transform_eval)
ds_test  = ImageSubset(images, labels, te, transform_eval)

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val, batch_size=32, shuffle=False, num_workers=0)
dl_test  = DataLoader(ds_test, batch_size=32, shuffle=False, num_workers=0)

x, y = next(iter(dl_train))
print("Batch :", x.shape, y.shape, x.dtype)

Batch : torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.float32


## 5. Deux configurations : euclidienne vs courbe

Mêmes dimensions totales (192), seul le type de variété change.

In [7]:
def make_cfg_euclidean():
    return ModelCfg(
        name="euclid",
        manifold=ManifoldCfg(
            type="product",
            components=[{"type": "euclidean", "dim": 192}],
            pooling="mean",
        ),
    )


def make_cfg_curved(curvature: float = 1.0):
    return ModelCfg(
        name="curved",
        manifold=ManifoldCfg(
            type="product",
            components=[
                {"type": "euclidean", "dim": 64},
                {"type": "hyperbolic", "dim": 64, "curvature": curvature},
                {"type": "sphere", "dim": 64},
            ],
            pooling="tangent_mean",
        ),
    )


cfg_euclid = make_cfg_euclidean()
cfg_curved = make_cfg_curved(curvature=1.0)

print("Euclid dims :", sum(c["dim"] for c in cfg_euclid.manifold.components))
print("Curved dims :", sum(c["dim"] for c in cfg_curved.manifold.components))

Euclid dims : 192
Curved dims : 192


## 6. Boucle d'entraînement et évaluation

Fonctions génériques pour entraîner et évaluer un modèle.

In [8]:
def train_model(cfg, n_epochs: int = 15, lr: float = 1e-4, seed: int = 0):
    """Entraîne un modèle JEPA sur les 5 classes."""
    set_seed(seed)
    model = JepaModel(cfg).to(device)
    model.register_new_classes(list(range(n_classes)))

    opt = build_optimizer(model, lr=lr)

    def loss_fn(m, batch, out):
        xb, yb = batch
        return classification_loss(out.logits, yb.to(device))

    state = train_one_task(
        model=model,
        train_loader=dl_train,
        val_loader=dl_val,
        optimizer=opt,
        n_epochs=n_epochs,
        loss_fn=loss_fn,
        device=device,
        task_idx=0,
        verbose=True,
    )
    return model, state


@torch.no_grad()
def evaluate(model, loader, n_cls):
    model.eval()
    all_logits, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        if out.logits is not None:
            all_logits.append(out.logits.cpu())
            all_y.append(yb)
    if not all_logits:
        return 0.0, {}
    logits = torch.cat(all_logits, 0)
    y = torch.cat(all_y, 0)
    acc = float((logits.argmax(-1) == y).float().mean().item())
    per_cls = per_class_accuracy(logits, y, n_cls)
    return acc, {c: float(per_cls[c]) for c in range(n_cls)
                 if not np.isnan(per_cls[c])}

## 7. Entraînement des deux modèles

⚠️ **Durée estimée** : 15-25 min sur CPU, 3-5 min sur GPU.

In [ ]:
N_EPOCHS = 15

print("=" * 60)
print("Modèle EUCLIDIEN (sans courbure)")
print("=" * 60)
model_e, state_e = train_model(cfg_euclid, n_epochs=N_EPOCHS, lr=1e-4, seed=0)

print("\n" + "=" * 60)
print("Modèle COURBE (E×H×S)")
print("=" * 60)
model_c, state_c = train_model(cfg_curved, n_epochs=N_EPOCHS, lr=1e-4, seed=0)

print("\nEntraînement terminé.")

Modèle EUCLIDIEN (sans courbure)


Using cache found in C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


11:32:21 | INFO    | src.models.backbone | Backbone tronqué : 10 blocs sur 12
11:32:21 | INFO    | src.models.backbone | Backbone gelé


[Tâche 0] époques:   0%|          | 0/15 [00:00<?, ?it/s]

  epoch 1/15:   0%|          | 0/266 [00:00<?, ?it/s]

## 8. Évaluation in-distribution

In [ ]:
acc_e_id, per_e_id = evaluate(model_e, dl_test, n_classes)
acc_c_id, per_c_id = evaluate(model_c, dl_test, n_classes)

print(f"Euclidien (ID) : acc={acc_e_id:.4f}")
print(f"Courbe    (ID) : acc={acc_c_id:.4f}")
print(f"Delta          : {acc_c_id - acc_e_id:+.4f}")

print("\nPar classe :")
for c in range(n_classes):
    name = loader.name_label_[c]
    a_e = per_e_id.get(c, np.nan)
    a_c = per_c_id.get(c, np.nan)
    print(f"  {name:12s} : euclid={a_e:.3f} | courbe={a_c:.3f} | delta={a_c - a_e:+.3f}")

## 9. Jeu OOD

Deux options : un second zip (`data/ood.zip`) ou des corruptions du test.

In [ ]:
OOD_ZIP = ROOT / "data" / "ood.zip"
USE_OOD_ZIP = OOD_ZIP.exists()

if USE_OOD_ZIP:
    with zipfile.ZipFile(OOD_ZIP, "r") as zf:
        with zf.open("loader.pkl") as f:
            loader_ood = pickle.load(f, encoding="latin1")
    if loader_ood.data_ is None:
        loader_ood.data_ = copy.deepcopy(loader_ood.original_data_)
    print("OOD zip trouvé :", loader_ood.data_.shape)
else:
    print("Pas de ood.zip — on utilisera les corruptions.")

In [ ]:
class CorruptTransform:
    """Applique une transformation OOD à une image PIL."""
    def __init__(self, kind="blur", severity=2):
        self.kind = kind
        self.severity = severity

    def __call__(self, img):
        from PIL import Image, ImageFilter, ImageEnhance
        if self.kind == "blur":
            return img.filter(ImageFilter.GaussianBlur(radius=self.severity))
        if self.kind == "brightness":
            return ImageEnhance.Brightness(img).enhance(1.0 + 0.3 * self.severity)
        if self.kind == "contrast":
            return ImageEnhance.Contrast(img).enhance(max(0.1, 1.0 - 0.2 * self.severity))
        if self.kind == "noise":
            arr = np.array(img).astype(np.int16)
            noise = np.random.randint(-20 * self.severity, 20 * self.severity,
                                      arr.shape, dtype=np.int16)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            return Image.fromarray(arr)
        return img


CORRUPTION = "blur"
SEVERITY = 2

if not USE_OOD_ZIP:
    ood_transform = T.Compose([
        T.Resize(256),
        CorruptTransform(kind=CORRUPTION, severity=SEVERITY),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    ds_ood = ImageSubset(images, labels, te, transform=ood_transform)
    dl_ood = DataLoader(ds_ood, batch_size=32, shuffle=False, num_workers=0)
    print(f"OOD corrompu : {CORRUPTION} (severity={SEVERITY})")
else:
    ood_imgs = loader_ood.data_["Image"].tolist()
    ood_lbls = np.array([loader.class_mapping_.get(l, -1)
                         for l in loader_ood.data_["Label"].tolist()])
    valid = ood_lbls >= 0
    ds_ood = ImageSubset(ood_imgs, ood_lbls, np.where(valid)[0], transform_eval)
    dl_ood = DataLoader(ds_ood, batch_size=32, shuffle=False, num_workers=0)
    print(f"OOD réel : {len(ds_ood)} images")

## 10. Évaluation OOD

In [ ]:
acc_e_ood, per_e_ood = evaluate(model_e, dl_ood, n_classes)
acc_c_ood, per_c_ood = evaluate(model_c, dl_ood, n_classes)

print(f"Euclidien (OOD) : acc={acc_e_ood:.4f}")
print(f"Courbe    (OOD) : acc={acc_c_ood:.4f}")
print(f"Delta           : {acc_c_ood - acc_e_ood:+.4f}")

print("\nDégradation (ID − OOD) :")
print(f"  Euclidien : {acc_e_id - acc_e_ood:+.4f}")
print(f"  Courbe    : {acc_c_id - acc_c_ood:+.4f}")

## 11. Synthèse et comparaison

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Modèle":      ["Euclidien", "Courbe"],
    "Acc ID":      [acc_e_id, acc_c_id],
    "Acc OOD":     [acc_e_ood, acc_c_ood],
    "Dégradation": [acc_e_id - acc_e_ood, acc_c_id - acc_c_ood],
    "Δ_ID":        [0.0, acc_c_id - acc_e_id],
    "Δ_OOD":       [0.0, acc_c_ood - acc_e_ood],
})

print(summary.to_string(index=False, float_format="%.4f"))

out_dir = ROOT / "results"
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / "curvature_generalization.csv", index=False)
print(f"\nSauvegardé : {out_dir / 'curvature_generalization.csv'}")

## 12. Figure : ID vs OOD

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

x = np.arange(2)
width = 0.35
vals_id = [acc_e_id, acc_c_id]
vals_ood = [acc_e_ood, acc_c_ood]

bars1 = ax.bar(x - width/2, vals_id, width, label="In-distribution",
               color="#0072B2", edgecolor="white", linewidth=1.5)
bars2 = ax.bar(x + width/2, vals_ood, width, label="OOD",
               color="#D55E00", edgecolor="white", linewidth=1.5)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f"{h:.3f}", ha="center", va="bottom",
                fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(["Euclidien", "Courbe"])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Impact de la courbure sur la généralisation", loc="left")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(out_dir / "curvature_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## 13. Figure : accuracy par classe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, (per_e, per_c, title) in zip(
    axes,
    [(per_e_id, per_c_id, "In-distribution"),
     (per_e_ood, per_c_ood, "OOD")],
):
    x = np.arange(n_classes)
    width = 0.35
    vals_e = [per_e.get(c, 0.0) for c in range(n_classes)]
    vals_c = [per_c.get(c, 0.0) for c in range(n_classes)]

    ax.bar(x - width/2, vals_e, width, label="Euclidien",
           color="#0072B2", edgecolor="white", linewidth=1.2)
    ax.bar(x + width/2, vals_c, width, label="Courbe",
           color="#D55E00", edgecolor="white", linewidth=1.2)

    ax.set_xticks(x)
    ax.set_xticklabels(loader.name_label_, rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy" if ax is axes[0] else "")
    ax.set_title(title, loc="left")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Accuracy par classe : euclidien vs courbe",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "curvature_per_class.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. Figure : courbes d'apprentissage

In [ ]:
def plot_hist(hist, label, color, ax_loss, ax_acc):
    epochs = [h["epoch"] + 1 for h in hist]
    loss = [h["train_loss"] for h in hist]
    acc = [h.get("train_acc", np.nan) for h in hist]
    ax_loss.plot(epochs, loss, marker="o", linewidth=2, color=color, label=label)
    ax_acc.plot(epochs, acc, marker="o", linewidth=2, color=color, label=label)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_hist(state_e.history, "Euclidien", "#0072B2", ax1, ax2)
plot_hist(state_c.history, "Courbe",    "#D55E00", ax1, ax2)

for ax, ylabel, title in [(ax1, "Train loss", "Loss"),
                          (ax2, "Train acc", "Accuracy")]:
    ax.set_title(title, loc="left")
    ax.set_xlabel("Époque")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(out_dir / "curvature_learning_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 15. Diagnostic géométrique des embeddings

In [ ]:
from src.geometry.diagnostics import geometry_report


@torch.no_grad()
def collect_emb(model, loader, n_max=512):
    model.eval()
    feats, ys = [], []
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        feats.append(out.global_embed.cpu())
        ys.append(yb)
        n += xb.size(0)
        if n >= n_max:
            break
    return torch.cat(feats, 0)[:n_max], torch.cat(ys, 0)[:n_max]


emb_e, y_e = collect_emb(model_e, dl_test)
emb_c, y_c = collect_emb(model_c, dl_test)

diag_e = geometry_report(emb_e, manifold=model_e.manifold)
diag_c = geometry_report(emb_c, manifold=model_c.manifold)

print("Euclidien :")
for k, v in diag_e.items():
    print(f"  {k:30s} = {v:.4f}")

print("\nCourbe :")
for k, v in diag_c.items():
    print(f"  {k:30s} = {v:.4f}")

## 16. Figure : t-SNE des deux modèles

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (emb, y_true, title) in zip(
    axes,
    [(emb_e, y_e, "Euclidien"), (emb_c, y_c, "Courbe")],
):
    proj = TSNE(n_components=2, perplexity=30, random_state=0,
                init="pca").fit_transform(emb.numpy())
    for c in range(n_classes):
        m = (y_true == c).numpy()
        if not m.any():
            continue
        ax.scatter(proj[m, 0], proj[m, 1], s=10, alpha=0.7,
                   label=loader.name_label_[c], edgecolors="none")
    ax.set_title(title, loc="left")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=8, loc="best", frameon=True, markerscale=1.8)

fig.suptitle("Projection t-SNE des embeddings sur le test",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "curvature_tsne.png", dpi=300, bbox_inches="tight")
plt.show()

## 17. Synthèse finale

In [ ]:
print("=" * 70)
print("  RÉSULTATS : impact de la courbure sur la généralisation")
print("=" * 70)
print(f"{'Métrique':<32s} {'Euclidien':>12s} {'Courbe':>12s} {'Δ':>10s}")
print("-" * 70)


def row(name, ve, vc):
    print(f"{name:<32s} {ve:>12.4f} {vc:>12.4f} {vc - ve:>+10.4f}")


row("Accuracy ID", acc_e_id, acc_c_id)
row("Accuracy OOD", acc_e_ood, acc_c_ood)
row("Dégradation ID→OOD", acc_e_id - acc_e_ood, acc_c_id - acc_c_ood)
row("δ-hyperbolicité", diag_e["delta_hyperbolicity"], diag_c["delta_hyperbolicity"])
row("Norme moyenne", diag_e["mean_norm"], diag_c["mean_norm"])
print("=" * 70)

print("\nInterprétation :")
if acc_c_ood > acc_e_ood + 0.02:
    print("  ✓ Le modèle courbe généralise mieux sur l'OOD.")
    print("    La courbure négative capture une structure hiérarchique")
    print("    utile face aux déformations inconnues.")
elif acc_c_ood < acc_e_ood - 0.02:
    print("  ✗ Le modèle euclidien généralise mieux sur l'OOD.")
    print("    La courbure n'apporte pas d'avantage ici, ou la déformation")
    print("    OOD est trop éloignée de la variété d'entraînement.")
else:
    print("  ≈ Pas de différence significative (|Δ| < 0.02).")
    print("    La courbure n'améliore ni ne dégrade la généralisation")
    print("    sur ce jeu OOD particulier.")

## Récapitulatif des fichiers produits

Dans `results/` :
- `curvature_generalization.csv` — tableau comparatif
- `curvature_comparison.png` — bar chart ID vs OOD
- `curvature_per_class.png` — accuracy par classe
- `curvature_learning_curves.png` — loss et acc pendant l'entraînement
- `curvature_tsne.png` — projection 2D des embeddings

**Protocole reproductible :**
- même backbone (DINOv2 gelé)
- même seed (0)
- même nombre d'époques (5)
- même learning rate (1e-3)
- seule différence : la tête variété (euclidienne vs courbe)